In [3]:
import pandas as pd

# Load the SMS Spam Collection dataset directly
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(
    url,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df.describe(include="all")

,label,message
count,5572,5572
unique,2,5169
top,ham,"Sorry, I'll call later"
freq,4825,30


In [5]:
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [6]:
print(df.isnull().sum())

label      0
message    0
dtype: int64


In [7]:
print(df.duplicated().sum())

403


In [9]:
print(df["label"].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [10]:
df=df.drop_duplicates()

In [11]:
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (5169, 2)


In [12]:
print(df["label"].value_counts())

label
ham     4516
spam     653
Name: count, dtype: int64


In [13]:
# Calculate the percentage of each label
print(df["label"].value_counts(normalize=True) * 100)

label
ham     87.366996
spam    12.633004
Name: proportion, dtype: float64


In [14]:
# Examine examples of legitimate messages
print("HAM EXAMPLES:")
print(df[df["label"] == "ham"]["message"].head(5))

# Examine examples of spam messages
print("\nSPAM EXAMPLES:")
print(df[df["label"] == "spam"]["message"].head(5))

HAM EXAMPLES:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
6    Even my brother is not like to speak with me. ...
Name: message, dtype: str

SPAM EXAMPLES:
2     Free entry in 2 a wkly comp to win FA Cup fina...
5     FreeMsg Hey there darling it's been 3 week's n...
8     WINNER!! As a valued network customer you have...
9     Had your mobile 11 months or more? U R entitle...
11    SIX chances to win CASH! From 100 to 20,000 po...
Name: message, dtype: str


In [18]:
# Separate the features and target
X = df["message"]
y = df["label"]

# Check their shapes
print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Inspect a few examples
print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())

Features shape: (5169,)
Target shape: (5169,)

Features:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: str

Target:
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: str


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

Training features: (4135,)
Testing features: (1034,)
Training targets: (4135,)
Testing targets: (1034,)


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the vectorizer
vectorizer = TfidfVectorizer()

# Learn vocabulary and calculate TF-IDF values from training messages
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform test messages using the learned vocabulary
X_test_tfidf = vectorizer.transform(X_test)

# Inspect the results
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("Number of learned words:", len(vectorizer.vocabulary_))

Training matrix shape: (4135, 7680)
Testing matrix shape: (1034, 7680)
Number of learned words: 7680


In [27]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create a simple baseline that always predicts the most frequent class
baseline = DummyClassifier(strategy="most_frequent")

# Train the baseline
baseline.fit(X_train_tfidf, y_train)

# Predict the test labels
y_baseline_pred = baseline.predict(X_test_tfidf)

# Evaluate the baseline
print("Baseline Accuracy:", accuracy_score(y_test, y_baseline_pred))

print("\nBaseline Classification Report:")
print(classification_report(y_test, y_baseline_pred, zero_division=0))

Baseline Accuracy: 0.8733075435203095

Baseline Classification Report:
              precision    recall  f1-score   support

         ham       0.87      1.00      0.93       903
        spam       0.00      0.00      0.00       131

    accuracy                           0.87      1034
   macro avg       0.44      0.50      0.47      1034
weighted avg       0.76      0.87      0.81      1034



In [28]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Create the model
nb_model = MultinomialNB()

# Train the model using our TF-IDF features
nb_model.fit(X_train_tfidf, y_train)

# Predict labels for unseen test messages
y_pred = nb_model.predict(X_test_tfidf)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9526112185686654

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       903
        spam       1.00      0.63      0.77       131

    accuracy                           0.95      1034
   macro avg       0.97      0.81      0.87      1034
weighted avg       0.96      0.95      0.95      1034

